# Verificación del Entorno y Versiones de Librerías en Google Colab

Este notebook inspecciona y reporta las versiones exactas de Python, del hardware asignado por Google Colab (GPU, RAM disponible y total), y de todas las librerías utilizadas a lo largo de este proyecto (incluyendo `data_utils.py` y todos los notebooks de las Fases 1, 2 y 3).

Genera una salida con el formato exacto requerido:
> `Python X.X; TensorFlow/Keras X.X; scikit-learn X.X; statsmodels/pmdarima X.X; granite-tsfm/transformers X.X; entorno Google Colab; GPU NVIDIA T4; RAM disponible`

seguido de una tabla detallada con el inventario completo de dependencias.

### 1. (Opcional) Instalación de dependencias del proyecto en Colab
> *Nota:* Ejecuta esta celda si te encuentras en una sesión limpia de Google Colab y deseas instalar los paquetes específicos del proyecto antes de comprobar las versiones.

In [ ]:
# Descomenta y ejecuta esta celda si acabas de iniciar una nueva sesión en Google Colab:
# !pip install -q pmdarima "granite-tsfm[notebooks]>=0.2" transformers>=4.40 datasets>=2.14 accelerate>=0.27 openmeteo-requests requests-cache retry-requests holidays

### 2. Detección y reporte de versiones
Inspecciona de forma segura todas las librerías utilizadas a lo largo del repositorio.

In [ ]:
import sys
import os
import platform
import importlib
import importlib.metadata

def get_pkg_version(dist_name, import_name=None):
    """Obtiene la versión instalada mediante metadata o inspeccionando __version__ de forma segura."""
    # 1. Buscar en metadata de distribuciones pip
    candidates = [dist_name, dist_name.replace("-", "_"), dist_name.replace("_", "-")]
    for name in candidates:
        try:
            return importlib.metadata.version(name)
        except Exception:
            pass
    # 2. Intentar importando el módulo y leyendo __version__
    mod_name = import_name or dist_name.replace("-", "_")
    try:
        mod = importlib.import_module(mod_name)
        return getattr(mod, "__version__", "instalado (sin __version__)")
    except Exception:
        return "no instalado"

# --- 1. Detección del entorno ---
try:
    import google.colab
    entorno = "Google Colab"
except ImportError:
    entorno = f"Local ({platform.system()} {platform.machine()})"

# --- 2. Detección de GPU ---
gpu_info = "Ninguna (CPU)"
try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        gpu_info = f"{name} ({vram:.1f} GB VRAM)"
except Exception:
    pass

if gpu_info == "Ninguna (CPU)":
    try:
        import subprocess
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
            text=True
        ).strip()
        if out:
            gpu_info = out.splitlines()[0].replace(", ", " (") + ")"
    except Exception:
        pass

# --- 3. Detección de RAM Disponible y Total ---
ram_disp_gb = None
ram_tot_gb = None
try:
    import psutil
    mem = psutil.virtual_memory()
    ram_disp_gb = mem.available / (1024**3)
    ram_tot_gb = mem.total / (1024**3)
except Exception:
    try:
        with open("/proc/meminfo", "r") as f:
            lines = f.readlines()
        mem_dict = {l.split(":")[0].strip(): l.split(":")[1].strip() for l in lines if ":" in l}
        total_kb = float(mem_dict.get("MemTotal", "0 kB").split()[0])
        avail_kb = float(mem_dict.get("MemAvailable", "0 kB").split()[0])
        ram_disp_gb = avail_kb / (1024**2)
        ram_tot_gb = total_kb / (1024**2)
    except Exception:
        pass

if ram_disp_gb is not None and ram_tot_gb is not None:
    ram_str = f"{ram_disp_gb:.1f} GB / {ram_tot_gb:.1f} GB total"
elif ram_disp_gb is not None:
    ram_str = f"{ram_disp_gb:.1f} GB"
else:
    ram_str = "N/D"

# --- 4. Versiones de todas las librerías del proyecto ---
# Extraídas de imports en data_utils.py y notebooks de Fases 1, 2 y 3
py_ver       = platform.python_version()
tf_ver       = get_pkg_version("tensorflow")
keras_ver    = get_pkg_version("keras")
sklearn_ver  = get_pkg_version("scikit-learn", "sklearn")
sm_ver       = get_pkg_version("statsmodels")
pmd_ver      = get_pkg_version("pmdarima")
tsfm_ver     = get_pkg_version("granite-tsfm", "tsfm_public")
transf_ver   = get_pkg_version("transformers")
torch_ver    = get_pkg_version("torch")
pd_ver       = get_pkg_version("pandas")
np_ver       = get_pkg_version("numpy")
scipy_ver    = get_pkg_version("scipy")
plt_ver      = get_pkg_version("matplotlib")
sns_ver      = get_pkg_version("seaborn")
datasets_ver = get_pkg_version("datasets")
accel_ver    = get_pkg_version("accelerate")
req_ver      = get_pkg_version("requests")
om_ver       = get_pkg_version("openmeteo-requests", "openmeteo_requests")
req_cache_ver= get_pkg_version("requests-cache", "requests_cache")
retry_ver    = get_pkg_version("retry-requests", "retry_requests")
hf_hub_ver   = get_pkg_version("huggingface-hub", "huggingface_hub")
gapi_ver     = get_pkg_version("google-api-python-client", "googleapiclient")
holidays_ver = get_pkg_version("holidays")

# Formato principal conciso solicitado por el usuario:
summary_output = (
    f"Python {py_ver}; "
    f"TensorFlow/Keras {tf_ver}/{keras_ver}; "
    f"scikit-learn {sklearn_ver}; "
    f"statsmodels/pmdarima {sm_ver}/{pmd_ver}; "
    f"granite-tsfm/transformers {tsfm_ver}/{transf_ver}; "
    f"PyTorch {torch_ver}; "
    f"entorno {entorno}; "
    f"GPU {gpu_info}; "
    f"RAM disponible {ram_str}"
)

print("=" * 85)
print("OUTPUT SOLICITADO:")
print("=" * 85)
print(summary_output)
print("=" * 85)

# Tabla detallada clasificada por módulo del proyecto
categories = [
    ("Entorno & Hardware", [
        ("Python", py_ver),
        ("Entorno", entorno),
        ("Sistema Operativo", f"{platform.system()} {platform.release()} ({platform.machine()})"),
        ("GPU asignada", gpu_info),
        ("RAM de sistema", ram_str),
    ]),
    ("Datos y Álgebra (data_utils.py & Fase 1 EDA)", [
        ("pandas", pd_ver),
        ("numpy", np_ver),
        ("scipy", scipy_ver),
    ]),
    ("Visualización (Fases 1, 2 y 3)", [
        ("matplotlib", plt_ver),
        ("seaborn", sns_ver),
    ]),
    ("Modelado Clásico (Fase 2.1 SARIMAX & Fase 3 Comparativa)", [
        ("scikit-learn (sklearn)", sklearn_ver),
        ("statsmodels", sm_ver),
        ("pmdarima", pmd_ver),
    ]),
    ("Deep Learning & Foundation Models (Fase 2.2 LSTM & Fase 2.3 TTM)", [
        ("tensorflow", tf_ver),
        ("keras", keras_ver),
        ("torch (PyTorch)", torch_ver),
        ("granite-tsfm (tsfm_public)", tsfm_ver),
        ("transformers", transf_ver),
        ("datasets", datasets_ver),
        ("accelerate", accel_ver),
    ]),
    ("Descarga de Datos & Cloud Sync (data_utils.py & Google Drive)", [
        ("requests", req_ver),
        ("openmeteo-requests", om_ver),
        ("requests-cache", req_cache_ver),
        ("retry-requests", retry_ver),
        ("huggingface-hub", hf_hub_ver),
        ("google-api-python-client", gapi_ver),
        ("holidays", holidays_ver),
    ]),
]

print("\nINVENTARIO DETALLADO DE LIBRERÍAS:")
print(f"{'Librería / Recurso':<35} | {'Versión / Estado':<45}")
print("-" * 85)
for cat_name, items in categories:
    print(f"\n[{cat_name}]")
    for name, ver in items:
        print(f"  {name:<33} | {ver:<45}")
print("=" * 85)
